In [1]:
# PySpark is the Spark API for Python. In this project, I use PySpark to initialize the SparkContext.   

from pyspark import SparkContext, SparkConf

from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, lit, avg,when, to_date, year,quarter

In [2]:
# importing and initializing findspark for easy spark setup.
import findspark

findspark.init()

The data sources were downloaded via terminal to my working directoty.

curl -o dataset1.csv 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-BD0225EN-SkillsNetwork/labs/data/dataset1.csv'


curl -o dataset2.csv 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-BD0225EN-SkillsNetwork/labs/data/dataset2.csv'

In [3]:
## Setting the environment variable to reference the java path

import os

#declaring the java environment variable
os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-22"

# Creating a SparkContext object

sc = SparkContext.getOrCreate()

# Creating a Spark Session

spark = SparkSession \
    .builder \
    .appName("Python Spark DataFrames basic example") \
    .getOrCreate()


##When I execute my code without stating below configuration setting time parse policy, an exception was encountred. Adding this config solved the issue.
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


In [ ]:
### checking the spark properties 
spark

In [5]:
# Loading the data into spark dataframe

df1 = spark.read.csv("C:\\Users\\Gbenga\\OneDrive\\Documents\\Pyspark_Project\\dataset1.csv", header=True, inferSchema= True)

df2 = spark.read.csv("C:\\Users\\Gbenga\\OneDrive\\Documents\\Pyspark_Project\\dataset2.csv", header=True, inferSchema= True)

In [ ]:
# viewing data frames schema
df1.printSchema()

df2.printSchema()

In [7]:
#cleaning the dataframes by dropping some columns

df1= df1.drop('description','location')
df2 = df2.drop('notes')

In [8]:
#Add new column year to df1 by extracting year from date column
df1= df1.withColumn('year', year(to_date('date_column','dd/MM/yyyy')))

#Add new column quarter to df2 by extracting quarter from transaction date column

df2= df2.withColumn('quarter', quarter(to_date('transaction_date','dd/MM/yyyy')))



In [ ]:
# Joining the two dataframes together on customer id fields

df = df1.join(df2, on= 'customer_id', how= 'inner' )


# renaming amount and value columns
df = df.withColumnRenamed('amount','transaction_amount')
df = df.withColumnRenamed('value','transaction_value')

df.show()

In [10]:
## Creating a temporary view called Transactions to allow spark sql operations.
df.createTempView("Transactions")

In [ ]:
## Filtering the merged data by transaction amount greater than 1500.
df_filter = spark.sql("select * from transactions where transaction_amount > 1500")

df_filter.show()

In [12]:
## Agrregation total sales by year
grouped_agg = df.groupBy('year').agg(sum('transaction_amount').alias('total_sales')).sort('year')

grouped_agg.show()

##grouped_agg.sort('total_sales').show()

+----+-----------+
|year|total_sales|
+----+-----------+
|2022|      29800|
|2023|      28100|
|2024|      25700|
|2025|      25700|
|2026|      25700|
|2027|      25700|
|2028|      25700|
|2029|      25700|
|2030|       9500|
+----+-----------+



In [ ]:
## adding a column based on benchmark of transaction amount greater than 5000

df=df.withColumn('benchmark', when(df['transaction_amount'] > 5000, lit('Yes')).otherwise(lit('No')))

df.show()

In [ ]:
## Average transaction amount per customer

customer_transactions = df.groupBy('customer_id').agg(avg('transaction_amount').alias('average_trans'))

## Writing the average customer transaction amount to hive table

customer_transactions.write.mode("overwrite").saveAsTable('customer_avg')